# 🔍 LangSmith Setup and Observability

## Learning Objectives
In this notebook, you will learn:
1. **LangSmith Tracing Basics** - How to enable automatic tracing for LangChain chains with a single environment variable
2. **Custom Run Naming** - How to tag and name traces for easier identification in the LangSmith dashboard
3. **Trace Metadata** - How to attach metadata (user IDs, request types) to traces for filtering and debugging
4. **Production Observability** - Why tracing matters for monitoring LLM applications in production

## Prerequisites
- Basic understanding of LangChain (LCEL chains, prompts, output parsers)
- A LangSmith account and API key ([sign up free](https://smith.langchain.com))
- A `.env` file at the project root with:
  - `OPENAI_API_KEY` - for OpenAI models
  - `LANGSMITH_API_KEY` - for LangSmith tracing

> Converted from `09_langsmith_setup.py` - part of **01 LangChain Foundations**.

---
## 📦 Part 1: Environment Setup and LangSmith Tracing Configuration

We load environment variables from `.env` and enable LangSmith tracing globally. Once tracing is enabled, every LangChain/LangGraph run made in this process is automatically logged to your LangSmith project - no extra instrumentation is needed inside each chain.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and LangSmith Tracing
# ============================================================================
import os

from dotenv import load_dotenv
from langsmith import traceable
from langsmith.run_trees import RunTree
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

load_dotenv()

# Enable LangSmith tracing for this process
# Requires LANGSMITH_API_KEY to be set in .env
os.environ["LANGSMITH_TRACING"] = "true"

if os.getenv("LANGSMITH_API_KEY"):
    print("✅ LangSmith tracing enabled and API key found!")
else:
    print("⚠️ LANGSMITH_TRACING is enabled but LANGSMITH_API_KEY is missing from .env")

---
## 🔭 Part 2: Tracing Demos

These three functions demonstrate progressively richer LangSmith tracing patterns: basic automatic tracing, named/tagged runs, and traces enriched with custom metadata. Each is decorated with `@traceable`, which wraps the function so LangSmith records its inputs, outputs, and nested LLM calls as a trace tree.

### 2.1 🧵 Basic Tracing

`demo_basic_tracing()` runs a simple prompt → LLM → parser chain. Because tracing is already enabled, this single call is automatically captured as a trace in your LangSmith project - no additional instrumentation is required.

In [ ]:
# ============================================================================
# TRACING DEMO: Basic Automatic Tracing
# ============================================================================
@traceable(name="basic_chaining")
def demo_basic_tracing():
    """Basic LangSmith tracing."""

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence.")

    chain = prompt | llm | StrOutputParser()

    print("Basic Tracing Demo:\n")
    print("Running chain with LangSmith tracing enabled...")

    result = chain.invoke({"topic": "machine learning"})

    print(f"Result: {result}")
    print("\nCheck LangSmith dashboard for trace details.")

### 2.2 🏷️ Named and Tagged Runs

`demo_named_runs()` shows how to give a run a custom `name` and `tags` via the `@traceable` decorator, making it easier to find and filter this specific chain's traces in the LangSmith UI.

In [ ]:
# ============================================================================
# TRACING DEMO: Named and Tagged Runs
# ============================================================================
@traceable(name="named_runs_demo", tags=["production", "summarization"])
def demo_named_runs():
    """Name your runs for easier identification."""

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    prompt = ChatPromptTemplate.from_template("Summarize: {text}")

    chain = prompt | llm | StrOutputParser()

    print("\nNamed Runs Demo:\n")

    result = chain.invoke(
        {"text": "LangSmith provides observability for LLM applications."}
    )

    print(f"Result: {result}")
    print("Run tagged with 'production', 'summarization'")

### 2.3 🗂️ Trace Metadata for Filtering

`demo_trace_with_metadata()` accepts a `user_id` and `request_type` and captures them as part of the trace context, so you can later filter or group traces in LangSmith by user or request type - useful for debugging a specific user's session in production.

In [ ]:
# ============================================================================
# TRACING DEMO: Custom Metadata
# ============================================================================
@traceable(name="trace_with_metadata_demo", tags=["metadata", "filtering"])
def demo_trace_with_metadata(user_id: str, request_type: str):
    """Add metadata to traces for filtering."""

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # Metadata is automatically captured
    result = llm.invoke(f"Hello from user {user_id}")

    return result.content

---
## ▶️ Part 3: Running the Demos

The block below is guarded by `if __name__ == "__main__":`, kept verbatim from the source script. Jupyter sets `__name__` to `"__main__"`, so this cell runs as-is when executed. Comment out any line to skip that particular demo.

In [ ]:
# ============================================================================
# RUN: Execute All Tracing Demos
# ============================================================================
if __name__ == "__main__":
    demo_basic_tracing()
    demo_named_runs()
    demo_trace_with_metadata(user_id="user_123", request_type="greeting")

---
## 📝 Summary

In this notebook, we learned:

### 1. Enabling LangSmith Tracing
- Setting `LANGSMITH_TRACING=true` (plus a valid `LANGSMITH_API_KEY` in `.env`) automatically traces every LangChain run made in the process
- No changes to your chains themselves are required

### 2. Tracing Patterns
- **Basic tracing**: `@traceable` on a function captures it and its nested LLM calls
- **Named/tagged runs**: pass `name=` and `tags=` to `@traceable` for easier identification in the dashboard
- **Metadata-enriched traces**: capture context like `user_id` or `request_type` for filtering and debugging

### Next Steps
- Open your LangSmith dashboard and inspect the traces produced by running this notebook
- Explore filtering traces by tag, name, and metadata in the LangSmith UI
- Move on to the next notebook to see production-grade observability patterns built on top of tracing